In [7]:
from dotenv import load_dotenv

load_dotenv()

from langgraph.graph import StateGraph, START, END
from typing import TypedDict, Literal, Annotated
from langchain_openai import ChatOpenAI
from langchain_core.messages import SystemMessage, HumanMessage
from pydantic import BaseModel, Field

In [8]:
generator_llm = ChatOpenAI(model='gpt-4o-mini')
evaluator_llm = ChatOpenAI(model='gpt-4o-mini')
optimizer_llm = ChatOpenAI(model='gpt-4o-mini')

In [9]:
class TweetEvaluation(BaseModel):
    evaluation: Literal["approved", "needs_improvement"] = Field(..., description="Final evaluation of the tweet")
    feedback: str = Field(..., description="Constructive feedback for the tweet")

In [10]:
structured_llm = evaluator_llm.with_structured_output(TweetEvaluation)

In [11]:
class TweetState(TypedDict):
    topic:str
    tweet:str
    evaluation:Literal["approved","needs_improvement"]
    feedback:str
    iteration:int
    max_iteration:int

In [12]:
def generate_tweet(state:TweetState):
    # prompt
    messages = [
        SystemMessage(content="You are a funny and clever Twitter/X influencer."),
        HumanMessage(content=f"""
        Write a short, original, and hilarious tweet on the topic: "{state['topic']}".
        
        Rules:
        - Do NOT use question-answer format.
        - Max 280 characters.
        - Use observational humor, irony, sarcasm, or cultural references.
        - Think in meme logic, punchlines, or relatable takes.
        - Use simple, day-to-day English.
        - This is version {state['iteration'] + 1}.
        """)
        ]

    response = generator_llm.invoke(messages).content

    return {'tweet':response}



In [13]:
def evaluate_tweet(state:TweetState):
    # prompt
    messages = [
        SystemMessage(content="You are a ruthless, no-laughs-given Twitter/X critic. You evaluate tweets honestly and help improve them."),
        HumanMessage(content=f"""
        Evaluate the following tweet:
        
        Tweet: "{state['tweet']}"
        
        Use the criteria below to evaluate the tweet:
        
        1. Originality - Is this fresh, or have you seen it a hundred times before?
        2. Humor - Did it genuinely make you smile, laugh, or chuckle?
        3. Punchiness - Is it short, sharp, and scroll-stopping?
        4. Virality Potential - Would people retweet or share it?
        5. Format - Is it a well-formed tweet, not a setup-punchline joke, not a Q&A joke, and under 280 characters?
        
        Auto-reject if:
        - It is written in question-answer format, such as "Why did..." or "What happens when..."
        - It exceeds 280 characters.
        - It reads like a traditional setup-punchline joke.
        - It ends with a generic, throwaway, or deflating line that weakens the humor.
        
        Respond ONLY in this structured format:
        - evaluation: "approved" or "needs_improvement"
        - feedback: Briefly explain the tweet's strengths and weaknesses.
        """)
            ]

    response = structured_llm.invoke(messages)

    return {'feedback':response.feedback,'evaluation':response.evaluation}

In [14]:
def optimize_tweet(state: TweetState):
    messages = [
        SystemMessage(content="You punch up tweets for virality and humor based on given feedback."),
        HumanMessage(content=f"""
        Improve the tweet based on this feedback:
        {state['feedback']}

        Topic: "{state['topic']}"
        Original Tweet:
        {state['tweet']}

        Re-write it as a short, viral-worthy tweet. Avoid Q&A style and stay under 280 characters.
        """)
    ]

    response = optimizer_llm.invoke(messages).content
    x = state['iteration'] + 1
    return {'tweet': response, 'iteration':x}

In [15]:
def check_condition(state: TweetState) -> Literal['approved', 'needs_improvement']:
    if state['evaluation'] == 'approved' or state['iteration'] >= state['max_iteration']:
        return 'approved'
    return 'needs_improvement'

In [16]:
graph = StateGraph(TweetState)

graph.add_node('generate', generate_tweet)
graph.add_node('evaluate', evaluate_tweet)
graph.add_node('optimize', optimize_tweet)

graph.add_edge(START, 'generate')
graph.add_edge('generate', 'evaluate')
graph.add_conditional_edges(
    'evaluate',
    check_condition,
    {'approved': END, 'needs_improvement': 'optimize'}
)
graph.add_edge('optimize', 'evaluate')

workflow = graph.compile()

In [17]:
initial_state = {
    "topic":"Indian Railways",
    "iteration":1,
    "max_iteration":10
}

final_state = workflow.invoke(initial_state)

print(final_state)

OpenAIRateLimitError: Error code: 429 - {'error': {'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, read the docs: https://platform.openai.com/docs/guides/error-codes/api-errors.', 'type': 'insufficient_quota', 'param': None, 'code': 'insufficient_quota'}}